# 03 – mes/batches.csv (MES)

**Goal:** understand this table well enough to explain it. Look and count only, no cleaning.

**What it is:** the batch records from the MES (Manufacturing Execution System). One batch = one production run on one line.

**Why it matters:** the assembly summary has **no batch ID**. To know which batch a filter assembly protected, we match assemblies to batches by **line + time window**. This file gives us those windows.

**Columns:** `batch_id`, `line`, `product`, `recipe`, `shift_crew`, `batch_start`, `batch_end`, `status`, `tubs_processed`, `total_good`, `rejects`, `comment`, `record_updated_at`.

**Done when I can answer without looking:**
1. What does one row represent?
2. What does each column mean (especially status and the two timestamps)?
3. What problems does the data have, and roughly how big is each?

**Questions to work through (from LEARNING_QA, "Data exploration"):** one row and key → packed columns → distinct values → number ranges → time → what the project needs → compare with `data/README.md`.

In [1]:
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## 1. Load all drops

In [2]:
# WHAT: load mes/batches.csv from every daily drop into one DataFrame, with the drop date.
# WHY:  each day's batch records arrive in a separate file; we need all 2 years together. The drop date tells us
#       when a record arrived, which matters because the README says each batch is delivered more than once.
#       Some drops may have no file (e.g. shutdowns), so check before reading.

drops_path = os.path.join("..", "data", "raw", "drops")

batch_list = []
for folder in sorted(os.listdir(drops_path)):              # folder = drop date, e.g. "2025-11-03"
    file_path = os.path.join(drops_path, folder, "mes", "batches.csv")
    if os.path.exists(file_path):
        df = pd.read_csv(file_path, dtype=str)              # read as text first; convert types later
        df["drop_date"] = folder
        batch_list.append(df)
    else:
        print("no batches.csv in", folder)

batches_df = pd.concat(batch_list, ignore_index=True)
print(len(batch_list), "files,", len(batches_df), "rows")
batches_df.head(10)

no batches.csv in 2024-12-27
no batches.csv in 2024-12-28
no batches.csv in 2024-12-29
no batches.csv in 2024-12-30
no batches.csv in 2024-12-31
no batches.csv in 2025-02-06
no batches.csv in 2025-04-20
no batches.csv in 2025-12-26
no batches.csv in 2025-12-27
no batches.csv in 2025-12-28
no batches.csv in 2025-12-29
no batches.csv in 2025-12-30
no batches.csv in 2025-12-31
no batches.csv in 2026-10-01
717 files, 2749 rows


,batch_id,line,product,recipe,shift_crew,batch_start,batch_end,status,tubs_processed,total_good,rejects,comment,record_updated_at,drop_date
0,B934425,LINE-A,PRD-215,RCP-215-v1,A-CREW1,2024-10-01T07:21:33Z,NaN,IN_PROGRESS,NaN,NaN,NaN,NaN,2024-10-01T07:21:33Z,2024-10-01
1,B593427,LINE-B,PRD-320,RCP-320-v1,B-CREW1,2024-10-01T06:30:22Z,NaN,IN_PROGRESS,NaN,NaN,NaN,NaN,2024-10-01T06:30:22Z,2024-10-01
2,B736530,LINE-C,PRD-412,RCP-412-v1,C-CREW1,2024-10-01T07:57:16Z,NaN,IN_PROGRESS,NaN,NaN,NaN,NaN,2024-10-01T07:57:16Z,2024-10-01
3,B934425,LINE-A,PRD-215,RCP-215-v1,A-CREW1,2024-10-01T07:21:33Z,2024-10-02T11:41:28Z,COMPLETED,2168,345394,1486,NaN,2024-10-02T11:41:28Z,2024-10-02
4,B686805,LINE-A,PRD-215,RCP-215-v1,A-CREW3,2024-10-02T19:40:28Z,NaN,IN_PROGRESS,NaN,NaN,NaN,NaN,2024-10-02T19:40:28Z,2024-10-02
5,B593427,LINE-B,PRD-320,RCP-320-v1,B-CREW1,2024-10-01T06:30:22Z,2024-10-02T08:28:06Z,COMPLETED,1779,282496,2144,NaN,2024-10-02T08:28:06Z,2024-10-02
6,B736530,LINE-C,PRD-412,RCP-412-v1,C-CREW1,2024-10-01T07:57:16Z,2024-10-02T11:17:23Z,COMPLETED,1876,298387,1773,NaN,2024-10-02T11:17:23Z,2024-10-02
7,B899576,LINE-C,PRD-412,RCP-412-v1,C-CREW1,2024-10-02T17:02:38Z,NaN,IN_PROGRESS,NaN,NaN,NaN,NaN,2024-10-02T17:02:38Z,2024-10-02
8,B558090,LINE-B,PRD-320,RCP-320-v1,B-CREW3,2024-10-03T00:45:59Z,NaN,IN_PROGRESS,NaN,NaN,NaN,NaN,2024-10-03T00:45:59Z,2024-10-03
9,B899576,LINE-C,PRD-412,RCP-412-v1,C-CREW1,2024-10-02T17:02:38Z,2024-10-03T19:55:51Z,COMPLETED,1898,302276,1404,NaN,2024-10-03T19:55:51Z,2024-10-03


In [9]:
# WHAT: count how many records each batch has, then summarise those counts.
# WHY:  the README says each batch arrives twice (IN_PROGRESS at start, final record at end). If that is true,
#       almost every batch has 2 records. Batches with 1 or 3+ records are the exceptions worth looking at.
#       Printing one count per batch (1,376 lines) is unreadable, so we count the counts instead.

records_per_batch = batches_df.groupby("batch_id").size()
print("distinct batches:", records_per_batch.shape[0])
print("batches by number of records:")
print(records_per_batch.value_counts().sort_index(), "\n")

# The exceptions: batches that do NOT have exactly 2 records
odd_batches = records_per_batch[records_per_batch != 2].index
batches_df[batches_df["batch_id"].isin(odd_batches)][["batch_id", "line", "status", "batch_start", "batch_end", "drop_date"]]

distinct batches: 1376
batches by number of records:
1       3
2    1373
Name: count, dtype: int64 



,batch_id,line,status,batch_start,batch_end,drop_date
157,B000000,LINE-B,TEST,2024-11-10T08:00:00Z,NaN,2024-11-10
1876,B000001,LINE-B,TEST,2026-02-13T08:00:00Z,NaN,2026-02-13
2302,NA-PENDING,LINE-B,NaN,2026-06-03T08:00:00Z,NaN,2026-06-03


In [10]:
# WHAT: convert the time columns to datetime, then calculate batch duration (end - start).
# WHY:  everything was read as text. Durations tell us how long a batch window is, which we need for matching
#       assemblies to batches. IN_PROGRESS records have no end time, so their duration is NaT (empty) - that is
#       expected, not an error.

for col in ["batch_start", "batch_end", "record_updated_at"]:
    batches_df[col] = pd.to_datetime(batches_df[col])   # values end in "Z" -> UTC-aware

batches_df["batch_duration"] = batches_df["batch_end"] - batches_df["batch_start"]
print("rows with a duration:", batches_df["batch_duration"].notna().sum(), "of", len(batches_df))

rows with a duration: 1373 of 2749


In [11]:
# WHAT: summarise batch durations per status (min, median, max).
# WHY:  .unique() lists 1,350 different durations, which tells us nothing. A summary shows the normal range
#       (DOMAIN_NOTES says ~1.5 days) and any odd values: very short (cancelled?) or very long batches.

batches_df.groupby("status")["batch_duration"].describe()[["count", "min", "50%", "max"]]

,count,min,50%,max
status,,,,
CANCELLED,13,0 days 00:07:14,0 days 13:45:08,0 days 14:41:36
COMPLETED,1360,0 days 18:51:20,1 days 05:39:54.500000,1 days 18:02:22
IN_PROGRESS,0,NaT,NaT,NaT
TEST,0,NaT,NaT,NaT


In [12]:
# WHAT: convert the count columns to numbers, then summarise tubs processed per batch.
# WHY:  the counts are text (dtype=str). Summing text JOINS the strings ("2168" + "1779" = "21681779") instead
#       of adding them - a silent wrong result. Only the final record (COMPLETED/CANCELLED) has counts; the
#       IN_PROGRESS record has none. So we look at final records only, one per batch, instead of summing.

for col in ["tubs_processed", "total_good", "rejects"]:
    batches_df[col] = pd.to_numeric(batches_df[col], errors="coerce")

final_records = batches_df[batches_df["status"].isin(["COMPLETED", "CANCELLED"])]
print("final records:", len(final_records), "| batches:", final_records["batch_id"].nunique())
final_records.groupby("status")[["tubs_processed", "total_good", "rejects"]].describe().T

final records: 1373 | batches: 1373


status                CANCELLED      COMPLETED
tubs_processed count       13.0    1360.000000
               mean         0.0    2404.855882
               std          0.0     338.624865
               min          0.0    1723.000000
               25%          0.0    2129.000000
               50%          0.0    2395.000000
               75%          0.0    2672.000000
               max          0.0    3184.000000
total_good     count       13.0    1360.000000
               mean         0.0  382851.162500
               std          0.0   53901.462557
               min          0.0  274367.000000
               25%          0.0  339458.000000
               50%          0.0  381248.500000
               75%          0.0  425271.000000
               max          0.0  506815.000000
rejects        count       13.0    1360.000000
               mean         0.0    1925.778676
               std          0.0     735.304794
               min          0.0     576.000000
               25%          0.0    1329.000000
               50%          0.0    1861.500000
               75%          0.0    2474.000000
               max          0.0    3907.000000

In [13]:
# WHAT: show the cancelled batches with their comment.
# WHY:  cancelled batches have 0 tubs and 0 good units, so it is tempting to drop them. The comment tells us
#       WHY they were cancelled. If the reason is a filter failure, these batches are exactly the deviations
#       the agent must investigate, so they must stay linked to their assemblies (decision D-012).

cancelled = final_records[final_records["status"] == "CANCELLED"].sort_values("batch_start")
cancelled[["batch_id", "line", "batch_start", "batch_end", "batch_duration", "comment"]]

,batch_id,line,batch_start,batch_end,batch_duration,comment
1764,B832481,LINE-A,2026-01-17 02:20:32+00:00,2026-01-17 15:42:41+00:00,0 days 13:22:09,Batch cancelled - filter integrity failure on ...
1776,B954264,LINE-C,2026-01-19 03:08:57+00:00,2026-01-19 16:11:39+00:00,0 days 13:02:42,Batch cancelled - filter integrity failure on ...
1781,B937224,LINE-A,2026-01-21 01:33:45+00:00,2026-01-21 14:27:35+00:00,0 days 12:53:50,Batch cancelled - filter integrity failure on ...
1793,B860242,LINE-C,2026-01-23 04:52:58+00:00,2026-01-23 19:26:34+00:00,0 days 14:33:36,Batch cancelled - filter integrity failure on ...
1797,B710195,LINE-C,2026-01-24 03:29:27+00:00,2026-01-24 16:57:27+00:00,0 days 13:28:00,Batch cancelled - filter integrity failure on ...
1802,B729977,LINE-C,2026-01-25 01:13:07+00:00,2026-01-25 15:20:37+00:00,0 days 14:07:30,Batch cancelled - filter integrity failure on ...
2047,B873940,LINE-B,2026-03-30 16:41:34+00:00,2026-03-31 06:02:10+00:00,0 days 13:20:36,Batch cancelled - filter integrity failure on ...
2052,B832151,LINE-B,2026-03-31 14:43:11+00:00,2026-04-01 05:12:18+00:00,0 days 14:29:07,Batch cancelled - filter integrity failure on ...
2055,B431419,LINE-C,2026-04-01 04:36:48+00:00,2026-04-01 04:44:02+00:00,0 days 00:07:14,Batch cancelled - filter integrity failure on ...
2059,B695853,LINE-C,2026-04-01 11:09:07+00:00,2026-04-02 00:54:15+00:00,0 days 13:45:08,Batch cancelled - filter integrity failure on ...


## 2. Checks for matching assemblies to batches
MES is used to match filter assemblies to batches by **line + time window**. What could break that?
1. Batches on the same line **overlap** → an assembly matches two batches.
2. **Gaps** between batches (changeover) → an assembly matches no batch.
3. **Missing files** → batch records lost?

In [14]:
# WHAT: check whether two batches on the same line ever overlap in time.
# WHY:  we will match an assembly to a batch by line + time window. If two batches on one line overlapped, an
#       assembly in the overlap would match BOTH, and we could not say which batch it protected.
#       Method: per line, sort batches by start time; each batch must start after the previous one ended.

final_sorted = final_records.sort_values(["line", "batch_start"]).copy()

# shift(1) within each line = "the row above" = the previous batch on the same line
final_sorted["previous_end"] = final_sorted.groupby("line")["batch_end"].shift(1)
final_sorted["gap_hours"] = (final_sorted["batch_start"] - final_sorted["previous_end"]).dt.total_seconds() / 3600

overlaps = final_sorted[final_sorted["gap_hours"] < 0]     # negative gap = started before the previous one ended
print("overlapping batches:", len(overlaps))
overlaps[["line", "batch_id", "batch_start", "previous_end", "gap_hours"]]

overlapping batches: 0


,line,batch_id,batch_start,previous_end,gap_hours


In [15]:
# WHAT: look at the gaps between batches on the same line (time from one batch's end to the next one's start).
# WHY:  during a gap (changeover) no batch is running. An assembly prepared in a gap matches NO batch window,
#       so the matching rule needs to handle it. We need the normal gap size, and the long gaps (shutdowns).

print("Gap in hours between batches (per line):")
print(final_sorted.groupby("line")["gap_hours"].describe()[["count", "min", "50%", "max"]], "\n")

# Long gaps (> 24 h): not a normal changeover. When do they happen?
long_gaps = final_sorted[final_sorted["gap_hours"] > 24]
long_gaps[["line", "previous_end", "batch_start", "gap_hours"]]

Gap in hours between batches (per line):
        count       min       50%         max
line                                         
LINE-A  453.0  4.160000  7.275000  155.382500
LINE-B  457.0  4.326111  7.365278  171.523056
LINE-C  460.0  4.292778  7.406667  175.953889 



,line,previous_end,batch_start,gap_hours
338,LINE-A,2024-12-26 10:14:31+00:00,2025-01-01 12:14:09+00:00,145.993889
879,LINE-A,2025-05-19 07:06:35+00:00,2025-05-22 06:44:04+00:00,71.624722
1706,LINE-A,2025-12-25 20:31:14+00:00,2026-01-01 07:54:11+00:00,155.382500
2273,LINE-A,2026-05-25 00:22:51+00:00,2026-05-27 08:34:57+00:00,56.201667
339,LINE-B,2024-12-26 11:11:08+00:00,2025-01-01 16:00:17+00:00,148.819167
994,LINE-B,2025-06-18 18:56:17+00:00,2025-06-21 13:12:29+00:00,66.270000
1707,LINE-B,2025-12-25 02:46:32+00:00,2026-01-01 06:17:55+00:00,171.523056
2388,LINE-B,2026-06-23 06:13:36+00:00,2026-06-26 12:36:33+00:00,78.382500
341,LINE-C,2024-12-25 15:49:01+00:00,2025-01-01 06:27:55+00:00,158.648333
1107,LINE-C,2025-07-19 06:09:08+00:00,2025-07-21 08:08:54+00:00,49.996111


In [16]:
# WHAT: for the 2 drops with no batches.csv (2025-02-06, 2025-04-20), check if any batch started or ended that day.
# WHY:  a missing file can mean two different things: (a) data was lost (a batch event happened but was never
#       delivered), or (b) nothing happened, so the MES had nothing to send. Only (a) is a data problem.
#       If no batch started or ended on those dates, the file is missing simply because there was nothing to send.

for missing_day in ["2025-02-06", "2025-04-20"]:
    started_that_day = batches_df["batch_start"].dt.strftime("%Y-%m-%d") == missing_day
    ended_that_day = batches_df["batch_end"].dt.strftime("%Y-%m-%d") == missing_day
    events = batches_df[started_that_day | ended_that_day]
    print(missing_day, "-> batch records starting or ending that day:", len(events))

2025-02-06 -> batch records starting or ending that day: 0
2025-04-20 -> batch records starting or ending that day: 0


## Findings (also in `DATA_FINDINGS.md`, mes/batches #1–#12)

**Load**
- 2,749 rows from 717 drops. File missing in 14 drops: same 12 as signals (Christmas shutdowns, 2026-10-01) plus 2025-02-06 and 2025-04-20. On those 2 days no batch started or ended → nothing to send, not data loss.

**Records per batch**
- 1,376 distinct batch IDs. 1,373 real batches have exactly 2 records (`IN_PROGRESS` + final) → keep the latest by `record_updated_at`.
- 3 are not real: `B000000`, `B000001` (`TEST`) and `NA-PENDING` (no status). All line B, start at exactly 08:00:00, no end → quarantine, exclude from matching.

**Status and duration**
- Final status: `COMPLETED` 1,360, `CANCELLED` 13.
- `COMPLETED` duration: 19 h to 1 d 18 h, median ~1 d 6 h.
- `CANCELLED`: 7 min to ~15 h, 0 tubs, 0 good, 0 rejects. **Every comment: "Batch cancelled - filter integrity failure on replacement assembly".** Clustered in Jan 2026 (6) and 30 Mar – 3 Apr 2026 (7).

**Counts per `COMPLETED` batch**
- Tubs: 1,723–3,184 (median ~2,400).
- Good units: 274,367–506,815 (median ~381,000).
- Rejects: 576–3,907 (median ~1,860, about 0.5% of good).
- Counts are text after loading: summing text joins strings with no error → convert to numbers first.

**Decision D-012:** match assemblies to `COMPLETED` **and** `CANCELLED` batches (cancelled ones are the filter failures to investigate); product impact uses `COMPLETED` only. Exclude `TEST` / `NA-PENDING`.

**Matching checks**
- **No overlaps:** on every line each batch starts after the previous one ended.
- **Changeover gaps:** usually 6–9 h (median ~7.3 h), shortest ~4 h. An assembly prepared in a changeover gap belongs to the **next** batch on that line.
- **Long gaps (> 24 h) are shutdowns:** year-end 145–176 h on all lines (ending 1 Jan 2025 and 1 Jan 2026); planned per-line shutdowns 50–78 h, twice per line, one line at a time. An assembly inside a shutdown is suspicious → flag.

**Late records (#12)**
- None: every record arrives in the drop of the day it was written (checked outside this notebook).
